In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score

print('import sucessfull')

import sucessfull


In [2]:
#import data
df = pd.read_csv('D:\ML_LEARNING\ML_PROJECTS\Indian_Railways_Failure_Prediction\Dataset\processed.csv')

In [3]:
# Ye columns future information hai (post-outcome), inhe drop karo
leak_cols = [
    'failure_type_Brake Failure', 'failure_type_Signal Failure',
    'failure_type_Track Defect', 'failure_type_Unknown',
    'failure_type_Wheel Defect',
    'failure_severity_High', 'failure_severity_Low',
    'failure_severity_Medium', 'failure_severity_Unknown'
]

x = df.drop(['maintenance_required'] + leak_cols, axis=1)
y = df['maintenance_required']

In [4]:
#train test split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [5]:
#standard scaler
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [13]:
#implementating grid search cv
models = {
    "Logistic Regression": LogisticRegression(),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(),
    "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier(),
    "SVC": SVC()
}

model_list = []
f1_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    eval_score = classification_report(y_test,y_pred)
    print(eval_score)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])
    f1_list.append(f1_score(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.80      0.95      0.87     20820
         1.0       0.81      0.44      0.57      9180

    accuracy                           0.80     30000
   macro avg       0.80      0.70      0.72     30000
weighted avg       0.80      0.80      0.78     30000

Logistic Regression
              precision    recall  f1-score   support

         0.0       0.76      0.90      0.82     20820
         1.0       0.61      0.34      0.44      9180

    accuracy                           0.73     30000
   macro avg       0.68      0.62      0.63     30000
weighted avg       0.71      0.73      0.70     30000

K-Neighbors Classifier
              precision    recall  f1-score   support

         0.0       0.81      0.79      0.80     20820
         1.0       0.56      0.59      0.57      9180

    accuracy                           0.73     30000
   macro avg       0.69      0.69      0.69     30000
weighted avg       0.74      0.

In [14]:
results_df = pd.DataFrame(
    list(zip(model_list, f1_list)), 
    columns=['Model Name', 'Eval']
).sort_values(by=["Eval"], ascending=False)

results_df

,Model Name,Eval
3,Random Forest Classifier,0.652782
4,XGBClassifier,0.650739
5,CatBoosting Classifier,0.646652
6,AdaBoost Classifier,0.602510
7,SVC,0.591613
0,Logistic Regression,0.573721
2,Decision Tree,0.572838
1,K-Neighbors Classifier,0.436021


In [15]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report
import warnings
warnings.filterwarnings("ignore")

# Har model ke liye hyperparameter grid
params = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1, 10, 100],
        "penalty": ["l2"],
        "solver": ["lbfgs"]
    },
    "K-Neighbors Classifier": {
        "n_neighbors": [3, 5, 7, 9, 11],
        "weights": ["uniform", "distance"]
    },
    "Decision Tree": {
        "criterion": ["gini", "entropy"],
        "max_depth": [5, 10, 15, None],
        "min_samples_split": [2, 5, 10]
    },
    "Random Forest Classifier": {
        "n_estimators": [100, 200, 300],
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5],
        "max_features": ["sqrt", "log2"]
    },
    "XGBClassifier": {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2]
    },
    "CatBoosting Classifier": {
        "depth": [4, 6, 8],
        "learning_rate": [0.01, 0.05, 0.1],
        "iterations": [200, 300]
    },
    "AdaBoost Regressor".replace("Regressor", "Classifier"): {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 1]
    },
    "SVC": {
        "C": [0.1, 1, 10],
        "kernel": ["rbf", "linear"],
        "gamma": ["scale", "auto"]
    }
}

models = {
    "Logistic Regression": LogisticRegression(),
    "K-Neighbors Classifier": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest Classifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier(),
    "CatBoosting Classifier": CatBoostClassifier(verbose=False),
    "AdaBoost Classifier": AdaBoostClassifier(),
    "SVC": SVC()
}

model_list = []
f1_list = []
best_params_dict = {}
trained_models = {}

for name, model in models.items():
    print(f"Tuning: {name}...")
    
    grid = GridSearchCV(
        estimator=model,
        param_grid=params[name],
        cv=3,              # 3-fold cross validation, speed ke liye
        scoring='f1',
        n_jobs=-1,          # sab CPU cores use karo, fast hoga
        verbose=0
    )
    
    grid.fit(x_train, y_train)
    
    best_model = grid.best_estimator_
    y_pred = best_model.predict(x_test)
    
    f1 = f1_score(y_test, y_pred)
    
    print(f"Best Params: {grid.best_params_}")
    print(classification_report(y_test, y_pred))
    print("-" * 60)
    
    model_list.append(name)
    f1_list.append(f1)
    best_params_dict[name] = grid.best_params_
    trained_models[name] = best_model

# Final comparison table
results_df = pd.DataFrame(
    list(zip(model_list, f1_list)), 
    columns=['Model Name', 'Eval']
).sort_values(by=["Eval"], ascending=False)

results_df

Tuning: Logistic Regression...
Best Params: {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}
              precision    recall  f1-score   support

         0.0       0.80      0.96      0.87     20820
         1.0       0.82      0.44      0.57      9180

    accuracy                           0.80     30000
   macro avg       0.81      0.70      0.72     30000
weighted avg       0.80      0.80      0.78     30000

------------------------------------------------------------
Tuning: K-Neighbors Classifier...
Best Params: {'n_neighbors': 3, 'weights': 'uniform'}
              precision    recall  f1-score   support

         0.0       0.76      0.87      0.81     20820
         1.0       0.54      0.36      0.44      9180

    accuracy                           0.71     30000
   macro avg       0.65      0.61      0.62     30000
weighted avg       0.69      0.71      0.69     30000

------------------------------------------------------------
Tuning: Decision Tree...
Best Params: {'crit

KeyboardInterrupt: 